## Template Visual Odometry

In [3]:
import cv2
from PIL import Image
import glob
import numpy as np
import cv2 as cv2
from numpy.linalg import inv, pinv
import matplotlib.pyplot as plt
import time as t

#### Load Pictures

In [11]:
#path_left = "C:/Users/danis/OneDrive/DTU/Perception for Autonomous Systems/EXAM/Data/Question_32/left/*.png" #assuming png
#path_right = "C:/Users/danis/OneDrive/DTU/Perception for Autonomous Systems/EXAM/Data/Question_32/right/*.png" #assuming png
path_right = "C:/Users/micha/OneDrive/Pulpit/DTU/S2/34759/exam/spring23/visual_odo/right/*.png"
path_left = "C:/Users/micha/OneDrive/Pulpit/DTU/S2/34759/exam/spring23/visual_odo/left/*.png"


#print(path_left)
#print(path_right)


images_left = []
for filename in glob.glob(path_left): 
    #im=Image.open(filename)
    im = cv2.imread(filename)
    #print(filename)
    images_left.append(im)
    
images_right = []
for filename in glob.glob(path_right): 
    #im=Image.open(filename)
    im = cv2.imread(filename)
    images_right.append(im)

print(f"img left: {len(images_left)}")
print(f"img right: {len(images_right)}")

img left: 81
img right: 81


#### Helper functions

In [12]:
# camera matrix

# Camera Matrix
# K = np.array([[7.188560e+02, 0.000000e+00, 6.071928e+02], [0, 7.188560e+02, 1.852157e+02], [0, 0, 1]])

def getK():
    return np.array([[7.188560e+02, 0.000000e+00, 6.071928e+02], [0, 7.188560e+02, 1.852157e+02], [0, 0, 1]])

def removeDuplicate(queryPoints, refPoints, radius=5):
    # remove duplicate points from new query points,
    for i in range(len(queryPoints)):
        query = queryPoints[i]
        xliml, xlimh = query[0] - radius, query[0] + radius
        yliml, ylimh = query[1] - radius, query[1] + radius
        inside_x_lim_mask = (refPoints[:, 0] > xliml) & (refPoints[:, 0] < xlimh)
        curr_kps_in_x_lim = refPoints[inside_x_lim_mask]

        if curr_kps_in_x_lim.shape[0] != 0:
            inside_y_lim_mask = (curr_kps_in_x_lim[:, 1] > yliml) & (curr_kps_in_x_lim[:, 1] < ylimh)
            curr_kps_in_x_lim_and_y_lim = curr_kps_in_x_lim[inside_y_lim_mask, :]
            if curr_kps_in_x_lim_and_y_lim.shape[0] != 0:
                queryPoints[i] = np.array([0, 0])
    return (queryPoints[:, 0] != 0)

In [17]:
surf = cv2.xfeatures2d.SIFT_create()

MATCH_NUM = 200

def extract_keypoints_surf(img1, img2, K, baseline):
    """
    use surf to detect keypoint features
    remember to include a Lowes ratio test
    """
    kp1, des1 = surf.detectAndCompute(img1,None)
    kp2, des2 = surf.detectAndCompute(img2,None)
    
    matcher = cv2.DescriptorMatcher_create(cv2.DescriptorMatcher_FLANNBASED)
    knn_matches = matcher.knnMatch(des1, des2, 2)
    
    #Filter matches using the Lowe's ratio test
    ratio_thresh = 0.7
    good_matches = []
    for m,n in knn_matches:
        if m.distance < ratio_thresh * n.distance:
            good_matches.append(m)
            
    good_matches = sorted(good_matches, key = lambda x:x.distance)
    good_matches = good_matches[:MATCH_NUM]
    
    matchesTrainIdx = [x.trainIdx  for x in good_matches]
    matchesQueryIdx = [x.queryIdx  for x in good_matches]
   
    match_points1 = [kp1[i].pt for i in matchesQueryIdx]
    match_points2 = [kp2[i].pt for i in matchesTrainIdx]
    
    p1 = np.float32(np.array(match_points1))
    p2 = np.float32(np.array(match_points2))

    ##### ############# ##########
    ##### Do Triangulation #######
    ##### ########################
    #project the feature points to 3D with triangulation
    
    #projection matrix for Left and Right Image
    M_left = K.dot(np.hstack((np.eye(3), np.zeros((3, 1)))))
    M_rght = K.dot(np.hstack((np.eye(3), np.array([[-baseline, 0, 0]]).T)))

    p1_flip = np.vstack((p1.T, np.ones((1, p1.shape[0]))))
    p2_flip = np.vstack((p2.T, np.ones((1, p2.shape[0]))))

    P = cv2.triangulatePoints(M_left, M_rght, p1_flip[:2], p2_flip[:2])

    # Normalize homogeneous coordinates (P->Nx4  [N,4] is the normalizer/scale)
    P = P / P[3]
    land_points = P[:3]

    return land_points.T, p1

def featureTracking(prev_img, next_img, prev_points, world_points):
    """
    Use OpenCV to find the prev_points from the prev_img in the next_img
    Remember to remove points that could not be found from prev_points, next_points, and world_points
    hint: status == 1
    """
    params = dict(winSize=(21, 21),
                 maxLevel=3,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
    

    next_points, status, _ = cv2.calcOpticalFlowPyrLK(prev_img, next_img, prev_points, nextPts=None, \
                                                      winSize=params["winSize"], maxLevel=params["maxLevel"], criteria=params["criteria"])
    
    next_points = np.squeeze(next_points[np.where(status[:,0]==1),:])
    prev_points = np.squeeze(prev_points[np.where(status[:,0]==1),:])
    world_points = np.squeeze(world_points[np.where(status[:,0]==1),:])

    return world_points, prev_points, next_points

def playImageSequence(left_img, right_img, K):

    baseline = 0.72;

    ##### ################################# #######
    ##### Get 3D points Using Triangulation #######
    ##### #########################################
    """
    Implement step 1.2 and 1.3
    Store the features in 'reference_2D' and the 3D points (landmarks) in 'landmark_3D'
    hint: use 'extract_keypoints_surf' above
    """
    landmark_3D, reference_2D = extract_keypoints_surf(left_img, right_img, K, baseline)
    
    # reference
    reference_img = left_img

    for i in range(1, len(images_left)):
        print('image: ', i)
        curImage = images_left[i]
        curImage_R = images_right[i]

        ##### ############################################################# #######
        ##### Calculate 2D and 3D feature correspndances in t=T-1, and t=T  #######
        ##### #####################################################################
        """
        Implement step 2.2)
        Remember this is a part of a loop, so the initial features are already
        provided in step 1)-1.3) outside the loop in 'reference_2D' and 'landmark_3D'
        """
         # the 2D landmarks at the current time = t
        landmark_3D, reference_2D, tracked_2Dpoints = featureTracking(reference_img, 
                                                                      curImage, 
                                                                      reference_2D,
                                                                      landmark_3D)
        #print(f'Tracked features: {np.shape(tracked_2Dpoints)}')
        ##### ################################# #######
        ##### Calculate relative pose using PNP #######
        ##### #########################################
        _, rvec, tvec, inliers = cv2.solvePnPRansac(landmark_3D,tracked_2Dpoints,K, None)
        #print(f'Solved Ransac: rvec={rvec}, tvec={tvec}')
    
        ##### ####################################################### #######
        ##### Get Pose and Tranformation Matrix in world coordionates #######
        ##### ###############################################################
        rot, _ = cv2.Rodrigues(rvec)
        tvec = -rot.T.dot(tvec)  # coordinate transformation, from camera to world. What is the XYZ of the camera wrt World
        inv_transform = np.hstack((rot.T, tvec))  # inverse transform. A tranform projecting points from the camera frame to the world frame
        
        ##### ################################# #######
        ##### Get 3D points Using Triangulation #######
        ##### #########################################
        # re-obtain the 3D points
        """
        Implement step 2.4)
        """
        landmark_3D_new, reference_2D_new = extract_keypoints_surf(curImage, curImage_R, K, baseline)
        #print(f'Extracted keypoints: reference_2D_new={np.shape(reference_2D_new)}')
        
        #Project the points from camera to world coordinates
        reference_2D = reference_2D_new.astype('float32')
        landmark_3D = inv_transform.dot(np.vstack((landmark_3D_new.T, np.ones((1, landmark_3D_new.shape[0])))))
        landmark_3D = landmark_3D.T

        ##### ####################### #######
        ##### Done, Next image please #######
        ##### ###############################
        reference_img = curImage

        ##### ################################## #######
        ##### Print stuff #######
        ##### ##########################################
                
        print("Position: x = {0:02f}m y = {1:02f}m z = {2:02f}m".format(float(tvec[0]), float(tvec[1]),
                                                                           float(tvec[2])))
        
        print("Orientation: axis angle x: {0:02f}, axis angle y: {1:02f}, axis angle z: {2:02f}".format(float(rvec[0]), float(rvec[1]),
                                                                           float(rvec[2])))

#### Run it

In [18]:
left_img = images_left[0]
right_img = images_right[0]

K = getK()

playImageSequence(left_img, right_img, K)

image:  1
Position: x = 0.010593m y = -0.003715m z = 0.998372m
Orientation: axis angle x: -0.001141, axis angle y: 0.000233, axis angle z: -0.001132
image:  2
Position: x = 0.032278m y = -0.010070m z = 1.988021m
Orientation: axis angle x: -0.002495, axis angle y: 0.001465, axis angle z: -0.002477
image:  3
Position: x = 0.041717m y = -0.025211m z = 2.974839m
Orientation: axis angle x: -0.003596, axis angle y: 0.003066, axis angle z: -0.004815
image:  4
Position: x = 0.055811m y = -0.041908m z = 3.949193m
Orientation: axis angle x: -0.004259, axis angle y: 0.005375, axis angle z: -0.005478
image:  5
Position: x = 0.071680m y = -0.063123m z = 4.925632m
Orientation: axis angle x: -0.004036, axis angle y: 0.007632, axis angle z: -0.006510
image:  6
Position: x = 0.068570m y = -0.079153m z = 5.902891m
Orientation: axis angle x: -0.003768, axis angle y: 0.010096, axis angle z: -0.005367
image:  7
Position: x = 0.060005m y = -0.102285m z = 6.883052m
Orientation: axis angle x: -0.003329, axis 